# Hypothesis Testing

Now that the metrics and tracking implementation plan are decided, we must perform a power analysis to derive the required sample size for our hypothesis testing. This can be calculated using our baseline OEC metrics and MDE of 2%. 

## Power Analysis

In [5]:
import math
import statsmodels.stats.power as power

# Assumed bassline OEC metrics and MDE
baseline_mean = 840.0  # Historic baseline minutes per user per week
baseline_std = 360.0  # Assumed behavioral standard deviation
mde_percent = 0.02  # 2% Minimum Detectable Effect

# Calculate the Absolute Detectable Effect and Effect Size (Cohen's d)
absolute_effect = baseline_mean * mde_percent
cohens_d = absolute_effect / baseline_std

print("--- Experiment Design Parameters ---")
print(f"Baseline Weekly Listening: {baseline_mean} minutes")
print(f"Target MDE Lift: {mde_percent * 100}% (+{absolute_effect:.2f} minutes)")
print(f"Calculated Effect Size (Cohen's d): {cohens_d:.4f}\n")

# Solve for sample size per variant (two-sided independent t-test)
analysis = power.TTestIndPower()
required_n_per_variant = analysis.solve_power(
    effect_size=cohens_d, 
    alpha=0.05, 
    power=0.80, 
    ratio=1.0, 
    alternative="two-sided"
)

print("--- Power Analysis Results ---")
print(f"Required sample size per variant: {math.ceil(required_n_per_variant)}")
print(f"Total users enrolled (1:1 Sample Ratio): {math.ceil(required_n_per_variant) * 2}")

--- Experiment Design Parameters ---
Baseline Weekly Listening: 840.0 minutes
Target MDE Lift: 2.0% (+16.80 minutes)
Calculated Effect Size (Cohen's d): 0.0467

--- Power Analysis Results ---
Required sample size per variant: 7210
Total users enrolled (1:1 Sample Ratio): 14420


## Dataset Validation

Next we'll load in the simulated dataset (generated via `data_simulation.py`) to check for Sample Ratio Mismatch (SRM). This will be done using a Chi-Square Goodness-of-Fit test. 

In [6]:
import pandas as pd
import scipy.stats as stats

# 1. Load the generated data file
df = pd.read_csv("data/simulated_experiment_log.csv")

# 2. Extract unique users per variant (Do not count daily rows, count unique people!)
observed_counts = df.groupby("variant")["user_id"].nunique().values
total_observed_users = observed_counts.sum()

# 3. Define our expected 50/50 split ratio
expected_counts = [total_observed_users / 2, total_observed_users / 2]

print("--- Sample Ratio Mismatch (SRM) Validation Check ---")
print(f"Total Unique Users Enrolled: {total_observed_users:,}")
print(f"Observed - Control: {observed_counts[0]:,} | Treatment: {observed_counts[1]:,}")
print(
    f"Expected - Control: {expected_counts[0]:,} | Treatment: {expected_counts[1]:,}\n"
)

# 4. Execute Chi-Square Goodness-of-Fit Test
chi2_stat, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"SRM Validation p-value: {p_value:.6f}")

# 5. Strategic Evaluation Rule
# In industry, an SRM check uses a super strict alpha of 0.001 to flag catastrophic pipeline bugs.
if p_value < 0.001:
    print(
        "🚨 CRITICAL WARNING: Sample Ratio Mismatch detected! The experiment assignment pipeline is corrupted."
    )
else:
    print(
        "✅ PASSED: No Sample Ratio Mismatch detected. Traffic allocation matches expectations."
    )

--- Sample Ratio Mismatch (SRM) Validation Check ---
Total Unique Users Enrolled: 14,420
Observed - Control: 7,301 | Treatment: 7,119
Expected - Control: 7,210.0 | Treatment: 7,210.0

Chi-Square Statistic: 2.2971
SRM Validation p-value: 0.129617
✅ PASSED: No Sample Ratio Mismatch detected. Traffic allocation matches expectations.


## Import via Mixpanel API
While the data generated from faker must be manually uploaded to BigQuery, we can utilize Mixpanel's API to import it directly.

In [8]:
import pandas as pd
import requests
import json
import time
import os

# 1. Project Configuration Token 🔑
MIXPANEL_TOKEN = "9beb249923566f891d98da68eafd71e0"

# 2. Robust Path Mapping (Finds file relative to notebook execution directory)
notebook_dir = os.getcwd()

# If your notebook lives in 'analytics/', 'simulated_experiment_log.csv' lives
# inside its sibling or direct child folder. Let's trace it safely:
if os.path.exists(os.path.join(notebook_dir, "data", "simulated_experiment_log.csv")):
    file_path = os.path.join(notebook_dir, "data", "simulated_experiment_log.csv")
elif os.path.exists(os.path.join(notebook_dir, "simulated_experiment_log.csv")):
    file_path = os.path.join(notebook_dir, "simulated_experiment_log.csv")
else:
    # Root level fallback
    file_path = os.path.join(
        os.path.dirname(notebook_dir),
        "analytics",
        "data",
        "simulated_experiment_log.csv",
    )

print(f"Targeting data file path: {file_path}")

# 3. Load Dataframe cleanly from disk
df = pd.read_csv(file_path)
print(f"Loaded {len(df):,} longitudinal user-day rows. Preparing payload mapping...")

# 4. Transform rows into clean standard Mixpanel JSON event payloads
mixpanel_events = []

for idx, row in df.iterrows():
    # Optimization filter: drop completely empty lines where no session activity occurred
    if row["total_minutes"] == 0 and row["w4_active_day"] == 0:
        continue

    # We map to a standard 'app_session_summary' telemetry action payload event block
    event_payload = {
        "event": "app_session_summary",
        "properties": {
            "token": MIXPANEL_TOKEN,
            "distinct_id": str(row["user_id"]),
            "experiment_variant": str(row["variant"]),
            "is_complier": int(row["is_complier"]),
            "experiment_day": int(row["experiment_day"]),
            "search_minutes": float(row["search_minutes"]),
            "autoplay_minutes": float(row["autoplay_minutes"]),
            "smart_queue_minutes": float(row["smart_queue_minutes"]),
            "total_minutes": float(row["total_minutes"]),
            "crashes": int(row["crashes"]),
            "w4_active_day": int(row["w4_active_day"]),
        },
    }
    mixpanel_events.append(event_payload)

# 5. Batch Stream over internet protocols via Mixpanel Track Endpoint API
# Mixpanel enforces a strict cap maximum of 2,000 events per single structured array post body
batch_size = 2000
url = "https://mixpanel.com"
headers = {"Content-Type": "application/json"}

print(
    f"Payload transformation completed. Uploading {len(mixpanel_events):,} event strings..."
)

for i in range(0, len(mixpanel_events), batch_size):
    batch = mixpanel_events[i : i + batch_size]

    # Fire secure HTTP post request payload straight to cloud servers
    response = requests.post(url, data=json.dumps(batch), headers=headers)

    if response.status_code != 200 or response.text == "0":
        print(
            f"❌ Critical Pipeline Failure at index {i}: Response text: {response.text}"
        )
        break

    if i % 20000 == 0:
        print(
            f" Ingested logs {i:,} through {min(i + batch_size, len(mixpanel_events)):,}..."
        )

    time.sleep(0.05)  # Safe rate-limiting throttle spacing to avoid free tier drops

print("\n--- 🏁 Mixpanel Ingestion Synchronized Successfully ---")

Targeting data file path: /Users/adam/VSCode/spotify-smart-queue-ab-test/analytics/data/simulated_experiment_log.csv
Loaded 398,008 longitudinal user-day rows. Preparing payload mapping...
Payload transformation completed. Uploading 397,880 event strings...
❌ Critical Pipeline Failure at index 0: Response text: Method Not Allowed


--- 🏁 Mixpanel Ingestion Synchronized Successfully ---
